# 3. Access and cache sources

Purpose: establish reproducible access to the Jouvence KG roots, OpenTargets source inventory, local cache, and LaminDB entrypoint without requiring macFUSE or LaminHub auth.

Prerequisites:
- Run from the repo root with `uv run jupyter lab` or execute cells through `uv run python`/Jupyter.
- Optional: `gcloud` authenticated for `gs://jouvencekb/kg/v2`.
- Optional: LaminDB authenticated for `jkobject/jouvencekb`.

Outputs:
- No canonical KG writes.
- Access/status tables for local cache, `/mnt/gcs` mount, GCS CLI, LaminDB auth probe, and OpenTargets release/dataset inventory.
- Copy commands for targeted local fallback samples under `.omoc/gcs-cache/kg-v2/`.

Expected runtime:
- Default sample mode: seconds; no network and no writes.
- With `JOUVENCE_NOTEBOOK_ALLOW_NETWORK=1` (legacy fallback: `TXGNN_NOTEBOOK_ALLOW_NETWORK=1`): OpenTargets FTP and GCS/Lamin probes may take tens of seconds.

Safe sample mode:
- `SAMPLE_MODE=True` and `ALLOW_NETWORK=False` by default. Heavy/full access actions are shown as commands, not executed.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    # Allows running the notebook from notebooks/ in Jupyter.
    candidate = REPO_ROOT.parent
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
sys.path.insert(0, str(REPO_ROOT))

SAMPLE_MODE = os.environ.get("JOUVENCE_NOTEBOOK_SAMPLE_MODE", os.environ.get("TXGNN_NOTEBOOK_SAMPLE_MODE", "1")) != "0"
ALLOW_NETWORK = os.environ.get("JOUVENCE_NOTEBOOK_ALLOW_NETWORK", os.environ.get("TXGNN_NOTEBOOK_ALLOW_NETWORK", "0")) == "1"
ALLOW_WRITES = os.environ.get("JOUVENCE_NOTEBOOK_ALLOW_WRITES", os.environ.get("TXGNN_NOTEBOOK_ALLOW_WRITES", "0")) == "1"
OPEN_TARGETS_RELEASE = os.environ.get("OPEN_TARGETS_RELEASE", "latest")
DEFAULT_LOCAL_CACHE_ROOT = Path(os.environ.get("JOUVENCE_VERIFIED_KG_ROOT", os.environ.get("TXGNN_VERIFIED_KG_ROOT", "/Users/jkobject/mnt/gcs/jouvencekb-kg/v2")))
LOCAL_CACHE_ROOT = Path(os.environ.get("JOUVENCE_LOCAL_CACHE_ROOT", os.environ.get("TXGNN_LOCAL_CACHE_ROOT", DEFAULT_LOCAL_CACHE_ROOT)))
DATA_ROOT = Path(os.environ.get("JOUVENCE_DATA_ROOT", os.environ.get("TXGNN_DATA_ROOT", REPO_ROOT / "data")))
OT_ROOT = Path(os.environ.get("JOUVENCE_OT_ROOT", os.environ.get("TXGNN_OT_ROOT", DATA_ROOT / "opentargets")))
KG_ROOT_URI = os.environ.get("JOUVENCE_KG_ROOT", os.environ.get("TXGNN_KG_ROOT", str(DATA_ROOT / "kg")))
CANONICAL_GCS_ROOT = "gs://jouvencekb/kg/v2"
CANONICAL_MOUNT_ROOT = Path(os.environ.get("JOUVENCE_VERIFIED_KG_ROOT", os.environ.get("TXGNN_VERIFIED_KG_ROOT", "/Users/jkobject/mnt/gcs/jouvencekb-kg/v2")))

print(json.dumps({
    "repo_root": str(REPO_ROOT),
    "sample_mode": SAMPLE_MODE,
    "allow_network": ALLOW_NETWORK,
    "allow_writes": ALLOW_WRITES,
    "open_targets_release": OPEN_TARGETS_RELEASE,
    "local_cache_root": str(LOCAL_CACHE_ROOT),
    "ot_root": str(OT_ROOT),
    "kg_root_uri": KG_ROOT_URI,
}, indent=2))


## Import project access helpers

The notebook calls existing project utilities where possible. OpenTargets release discovery uses `txdata_download`; KG storage inspection uses `manage_db.kg_storage`. If optional dependencies/auth are missing, the cell records a skip instead of blocking later notebooks.


In [ ]:
from txdata_download import get_latest_opentargets_release, get_opentargets_releases, list_opentargets_datasets
from manage_db.kg_storage import open_kg_root

try:
    import lamin as ln
except Exception as exc:  # auth/package may be absent in worker shells
    ln = None
    lamin_import_error = repr(exc)
else:
    lamin_import_error = None


## Access probes and fallbacks

Do not require macFUSE. Prefer direct local cache/GCS-copy fallbacks when `/mnt/gcs` is not available.


In [ ]:
def run_cmd(args: list[str], timeout: int = 15) -> dict:
    try:
        proc = subprocess.run(args, text=True, capture_output=True, timeout=timeout, check=False)
        return {"ok": proc.returncode == 0, "returncode": proc.returncode, "stdout": proc.stdout[-500:], "stderr": proc.stderr[-500:]}
    except Exception as exc:
        return {"ok": False, "error": repr(exc)}

access_rows = []
access_rows.append({"surface": "repo", "available": REPO_ROOT.exists(), "path_or_uri": str(REPO_ROOT), "note": "current checkout"})
access_rows.append({"surface": "local_cache", "available": LOCAL_CACHE_ROOT.exists(), "path_or_uri": str(LOCAL_CACHE_ROOT), "note": "targeted GCS-copy fallback"})
access_rows.append({"surface": "canonical_mount", "available": CANONICAL_MOUNT_ROOT.exists(), "path_or_uri": str(CANONICAL_MOUNT_ROOT), "note": "optional macFUSE/gcsfuse mount; not required"})
access_rows.append({"surface": "gcloud_cli", "available": shutil.which("gcloud") is not None, "path_or_uri": "gcloud", "note": "used for targeted sample copies"})

if ALLOW_NETWORK and shutil.which("gcloud"):
    probe = run_cmd(["gcloud", "storage", "ls", CANONICAL_GCS_ROOT + "/nodes/"], timeout=20)
    access_rows.append({"surface": "gcs_nodes_listing", "available": probe.get("ok", False), "path_or_uri": CANONICAL_GCS_ROOT + "/nodes/", "note": probe.get("stderr") or probe.get("stdout") or probe.get("error")})
else:
    access_rows.append({"surface": "gcs_nodes_listing", "available": False, "path_or_uri": CANONICAL_GCS_ROOT + "/nodes/", "note": "skipped; set JOUVENCE_NOTEBOOK_ALLOW_NETWORK=1 (legacy fallback: TXGNN_NOTEBOOK_ALLOW_NETWORK=1) to probe"})

pd.DataFrame(access_rows)


In [ ]:
fallback_commands = [
    f"mkdir -p {LOCAL_CACHE_ROOT}/nodes {LOCAL_CACHE_ROOT}/edges {LOCAL_CACHE_ROOT}/evidence",
    f"gcloud storage cp {CANONICAL_GCS_ROOT}/nodes/gene.parquet {LOCAL_CACHE_ROOT}/nodes/",
    f"gcloud storage cp {CANONICAL_GCS_ROOT}/nodes/disease.parquet {LOCAL_CACHE_ROOT}/nodes/",
    f"gcloud storage cp {CANONICAL_GCS_ROOT}/edges/disease_associated_gene.parquet {LOCAL_CACHE_ROOT}/edges/",
]
print("Targeted fallback cache commands (manual; not run by default):")
print("\n".join(fallback_commands))


## OpenTargets release and source inventory

Network listing is guarded. In fully offline sample mode, keep `OPEN_TARGETS_RELEASE="latest"` as a symbolic release and use the required dataset list for downstream notebooks.


In [ ]:
required_datasets = [
    "target", "disease", "drug_molecule", "biosample", "go", "reactome",
    "target_homologues", "interaction", "evidence", "drug_indication",
    "drug_mechanism_of_action", "target_essentiality", "disease_phenotype",
    "expression", "pharmacogenomics", "variant", "known_variant", "enhancer_to_gene",
]

if ALLOW_NETWORK:
    releases = get_opentargets_releases()
    selected_release = get_latest_opentargets_release() if OPEN_TARGETS_RELEASE == "latest" else OPEN_TARGETS_RELEASE
    available_datasets = list_opentargets_datasets(selected_release)
else:
    releases = []
    selected_release = OPEN_TARGETS_RELEASE
    available_datasets = []

inventory = pd.DataFrame({
    "dataset": required_datasets,
    "listed_in_release": [d in available_datasets or not available_datasets for d in required_datasets],
    "local_path": [str(OT_ROOT / d) for d in required_datasets],
    "local_exists": [(OT_ROOT / d).exists() for d in required_datasets],
})
print({"selected_release": selected_release, "release_count": len(releases), "dataset_count": len(available_datasets)})
inventory


## KG root inspection

`open_kg_root` is the storage abstraction used by builders. In sample mode, inspect local/cache paths only; remote `gs://` reads require credentials and can be expensive.


In [ ]:
kg_candidates = [
    ("configured", KG_ROOT_URI),
    ("local_cache", str(LOCAL_CACHE_ROOT)),
    ("canonical_mount", str(CANONICAL_MOUNT_ROOT)),
]
rows = []
for name, uri in kg_candidates:
    try:
        if uri.startswith("gs://") and not ALLOW_NETWORK:
            raise RuntimeError("skipped remote gs:// open in offline sample mode")
        if "://" not in uri and not Path(uri).exists() and not ALLOW_WRITES:
            raise RuntimeError("local KG root is absent and ALLOW_WRITES=0, so not creating it")
        kg = open_kg_root(uri)
        rows.append({"name": name, "uri": uri, "open_ok": True, "nodes": kg.list_nodes()[:10], "edges": kg.list_edges()[:10]})
    except Exception as exc:
        rows.append({"name": name, "uri": uri, "open_ok": False, "error": repr(exc)})
pd.DataFrame(rows)


## LaminDB auth probe

LaminDB is useful for registries/parity, but missing auth should not block KG Parquet reproduction. The intended authenticated command is `ln.connect("jkobject/jouvencekb")`.


In [ ]:
if ln is None:
    lamin_status = {"import_ok": False, "error": lamin_import_error, "connect_command": 'ln.connect("jkobject/jouvencekb")'}
elif not ALLOW_NETWORK:
    lamin_status = {"import_ok": True, "connect_skipped": True, "connect_command": 'ln.connect("jkobject/jouvencekb")'}
else:
    try:
        ln.connect("jkobject/jouvencekb")
        lamin_status = {"import_ok": True, "connect_ok": True}
    except Exception as exc:
        lamin_status = {"import_ok": True, "connect_ok": False, "error": repr(exc), "fallback": "Use Parquet/GCS cache path and rerun auth-gated Lamin notebooks later."}
lamin_status


## Validation

Cheap checks: required source list is non-empty, access status did not print credential material, and fallback commands point at the documented cache root.


In [ ]:
assert required_datasets, "required OpenTargets/source dataset list must not be empty"
assert LOCAL_CACHE_ROOT.exists() or str(LOCAL_CACHE_ROOT).startswith("gs://") or SAMPLE_MODE
joined_commands = "\n".join(fallback_commands)
assert "gs://jouvencekb/kg/v2" in joined_commands
assert not any(secret in joined_commands.lower() for secret in ["token", "password", "secret_key"])
print("Notebook 3 lightweight validation passed.")
